# Exercício 22 — notícias (Web) + **FAISS** + **RAG** (dois agentes)

1. **Recolha:** pesquisa DuckDuckGo e indexação local em `faiss_noticias_index/`.
2. **Análise:** conversa com recuperação semântica sobre esse índice.

Requer **`GOOGLE_API_KEY`** no `.env` na raiz e **rede** para a pesquisa. Opcional: `GEMINI_MODEL_EX22`.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env, override=False)
        break

HERE = Path.cwd().resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from agente_recolha_noticias import (
    config_recolha,
    criar_grafo_agente_recolha,
    ultima_resposta_assistente as ultima_recolha,
)

os.environ.setdefault("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", ""))

g_recolha = criar_grafo_agente_recolha()
cfg = config_recolha("demo-notebook")
msg = (
    "Pesquisa notícias de Portugal para hoje (manchetes e economia) "
    "e grava o resultado no índice FAISS. Se precisares, faz uma última consulta abrangente antes de gravar."
)
out = g_recolha.invoke({"messages": [HumanMessage(content=msg)]}, cfg)
print(ultima_recolha(out))

O índice FAISS foi atualizado com um resumo das notícias de Portugal, incluindo manchetes e economia.


In [3]:
from agente_analista_noticias_rag import (
    config_analista,
    criar_grafo_agente_analista,
    ultima_resposta_assistente as ultima_analista,
)

g_rag = criar_grafo_agente_analista()
cfg2 = config_analista("demo-notebook-rag")
pergunta = (
    "Com base nas notícias indexadas, resume os temas principais e indica "
    "dois riscos ou incertezas para um decisor. Usa a ferramenta de recuperação."
    "pesquise sobre esportes"
)
out2 = g_rag.invoke({"messages": [HumanMessage(content=pergunta)]}, cfg2)
print(ultima_analista(out2))

O índice de notícias contém menções a:

*   **Novo aeroporto em Portugal:** A divulgação de um relatório sobre o novo aeroporto.
*   **Encontro diplomático:** Visita de Marcelo Rebelo de Sousa ao Brasil para encontrar-se com Lula da Silva.
*   **Mercado de jogos de azar online:** Crescimento do setor em Portugal.
*   **Substituição de profissionais:** Dificuldades na substituição de magistrados, professores e médicos que regressam a Portugal (contexto geográfico incerto).

**Riscos e Incertezas para um Decisor:**

1.  **Atrasos ou Imprevistos no Novo Aeroporto:** A construção de um novo aeroporto é um projeto complexo, sujeito a atrasos, estouros orçamentários e desafios ambientais. Um decisor deve estar preparado para mitigar esses riscos e comunicar eficazmente quaisquer problemas ao público.
2.  **Impacto Social do Jogo Online:** O crescimento do mercado de jogos de azar online pode trazer consigo problemas de vício, endividamento e outros impactos sociais negativos. Um decisor deve